# Laboratorium 5 (4 pkt)

Celem czwartego laboratorium jest zapoznanie się oraz zaimplementowanie algorytmów głębokiego uczenia aktywnego. Zaimplementowane algorytmy będą testowane z wykorzystaniem środowiska z OpenAI - *CartPole*.


Dołączenie standardowych bibliotek

In [12]:
from collections import deque
import gym
import numpy as np
import random
from copy import deepcopy

if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

Dołączenie bibliotek do obsługi sieci neuronowych

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim

class DQNModel(nn.Module):
    def __init__(self, input_dim, output_dim, learning_rate, hidden_units=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, output_dim),
        )
        self.loss_fn = nn.MSELoss()
        self.optimizer = optim.AdamW(self.parameters(), lr=learning_rate)

    def forward(self, x):
        return self.net(x)

    def predict(self, state):
        self.eval()
        with torch.no_grad():
            state_tensor = torch.as_tensor(state, dtype=torch.float32)
            q_values = self(state_tensor).cpu().numpy()
        return q_values

    def fit(self, states, targets):
        self.train()
        states_tensor = torch.as_tensor(states, dtype=torch.float32)
        targets_tensor = torch.as_tensor(targets, dtype=torch.float32)
        
        self.optimizer.zero_grad()
        predictions = self(states_tensor)
        loss = self.loss_fn(predictions, targets_tensor)
        loss.backward()
        self.optimizer.step()
        return loss.item()

## Zadanie 1 - Double Deep Q-Network

<p style='text-align: justify;'>
Celem ćwiczenie jest zaimplementowanie algorytmu Double Deep Q-Network. Wartoscią oczekiwaną sieci jest:
\begin{equation}
       Q^*(s, a) \approx r + \gamma argmax_{a'}Q_\theta'(s', a') 
\end{equation}
a wagi pomiędzy sieciami wymieniane są co dziesięć aktualizacji wag sieci sterującej poczynaniami agenta ($Q$).
</p>

In [14]:
class DDQNAgent:
    def __init__(self, state_size, action_size, model):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95    # discount rate
        self.epsilon = 0.5  # exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.95
        self.learning_rate = 0.001
        self.model = self._build_model(model)
        self.target_model = self._build_model(model)
        self.update_weights()
        self.replay_counter = 1

    def _build_model(self, model):
        return deepcopy(model)

    def remember(self, state, action, reward, next_state, done):
        #Function adds information to the memory about last action and its results
        self.memory.append((state, action, reward, next_state, done)) 

    def get_action(self, state):
        """
        Compute the action to take in the current state, including exploration.
        With probability self.epsilon, we should take a random action.
            otherwise - the best policy action (self.get_best_action).

        Note: To pick randomly from a list, use random.choice(list).
              To pick True or False with a given probablity, generate uniform number in [0, 1]
              and compare it with your probability
        """

        #
        # INSERT CODE HERE to get action in a given state (according to epsilon greedy algorithm)
        if random.uniform(0, 1) < self.epsilon:
            chosen_action = random.choice(range(self.action_size))
        else:
            chosen_action = self.get_best_action(state)
        #        
        
        return chosen_action

  
    def get_best_action(self, state):
        """
        Compute the best action to take in a state.
        """

        #
        # INSERT CODE HERE to get best possible action in a given state (remember to break ties randomly)
        q_values = self.model.predict(state)
        best_value = np.max(q_values)
        best_indices = np.flatnonzero(q_values == best_value)
        best_action = int(np.random.choice(best_indices))
        #

        return best_action

    def replay(self, batch_size):
        """
        Function learn network using randomly selected actions from the memory. 
        First calculates Q value for the next state and choose action with the biggest value.
        Target value is calculated according to:
                Q(s,a) := (r + gamma * max_a(Q(s', a)))
        except the situation when the next action is the last action, in such case Q(s, a) := r.
        In order to change only those weights responsible for chosing given action, the rest values should be those
        returned by the network for state state.
        The network should be trained on batch_size samples.
        After each 10 Q Network trainings parameters should be copied to the target Q Network
        """
        #
        # INSERT CODE HERE to train network
        mini_batch = random.sample(self.memory, batch_size)
        states = np.vstack([sample[0] for sample in mini_batch])
        next_states = np.vstack([sample[3] for sample in mini_batch])
        
        targets = self.model.predict(states)
        next_q = self.model.predict(next_states)

        for i, (_, action, reward, _, done) in enumerate(mini_batch):
            if done:
                target_value = reward
            else:
                target_value = reward + self.gamma * np.max(next_q[i])
            targets[i][action] = target_value

        self.model.fit(states, targets)

        self.replay_counter += 1
        if self.replay_counter % 10 == 0:
            self.update_weights()
        #

    def update_epsilon_value(self):
        #Every each epoch epsilon value should be updated according to equation: 
        #self.epsilon *= self.epsilon_decay, but the updated value shouldn't be lower then epsilon_min value
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def update_weights(self):
        """copy trained Q Network params to target Q Network"""
        #
        self.target_model = deepcopy(self.model)
        # INSERT CODE HERE to train network
        #


Czas przygotować model sieci, która będzie się uczyła działania w środowisku [*CartPool*](https://gym.openai.com/envs/CartPole-v0/):

In [15]:
env = gym.make("CartPole-v0").env
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
learning_rate = 0.001

model = DQNModel(state_size, action_size, learning_rate)

d:\.Astudia\.venv\Lib\site-packages\gym\envs\registration.py:555: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(


Czas nauczyć agenta gry w środowisku *CartPool*:

In [16]:
agent = DDQNAgent(state_size, action_size, model)

agent.epsilon = 0.75
agent.epsilon_decay = 0.9

done = False
batch_size = 64
EPISODES = 1000
counter = 0
for e in range(EPISODES):
    summary = []
    for _ in range(100):
        total_reward = 0
        reset_out = env.reset()
        env_state = reset_out[0] if isinstance(reset_out, tuple) else reset_out
    
        #
        # INSERT CODE HERE to prepare appropriate format of the state for network
        state = np.reshape(env_state, (1, state_size)).astype(np.float32)
        #
        
        for time in range(500):
            action = agent.get_action(state)
            step_out = env.step(action)
            if isinstance(step_out, tuple) and len(step_out) == 5:
                next_state_env, reward, terminated, truncated, _ = step_out
                done = terminated or truncated
            else:
                next_state_env, reward, done, _ = step_out
            total_reward += reward

            #
            # INSERT CODE HERE to prepare appropriate format of the next state for network
            next_state = np.reshape(next_state_env, (1, state_size)).astype(np.float32)
            #

            #add to experience memory
            agent.remember(state, action, reward, next_state, done)
            state = next_state
            if done:
                break

        #
        # INSERT CODE HERE to train network if in the memory is more samples then size of the batch
        if len(agent.memory) > batch_size:
            agent.replay(batch_size)
        #
        
        summary.append(total_reward)
        
    agent.update_epsilon_value()
    print("epoch #{}\tmean reward = {:.3f}\tepsilon = {:.3f}".format(e, np.mean(summary), agent.epsilon))    
    
    if np.mean(summary) > 195:
        print ("You Win!")
        break


epoch #0	mean reward = 18.230	epsilon = 0.675
epoch #1	mean reward = 15.280	epsilon = 0.608
epoch #2	mean reward = 15.140	epsilon = 0.547
epoch #3	mean reward = 30.770	epsilon = 0.492
epoch #4	mean reward = 53.300	epsilon = 0.443
epoch #5	mean reward = 52.210	epsilon = 0.399
epoch #6	mean reward = 52.740	epsilon = 0.359
epoch #7	mean reward = 76.520	epsilon = 0.323
epoch #8	mean reward = 93.950	epsilon = 0.291
epoch #9	mean reward = 83.920	epsilon = 0.262
epoch #10	mean reward = 87.960	epsilon = 0.235
epoch #11	mean reward = 100.760	epsilon = 0.212
epoch #12	mean reward = 118.030	epsilon = 0.191
epoch #13	mean reward = 118.200	epsilon = 0.172
epoch #14	mean reward = 139.170	epsilon = 0.154
epoch #15	mean reward = 157.990	epsilon = 0.139
epoch #16	mean reward = 181.760	epsilon = 0.125
epoch #17	mean reward = 136.960	epsilon = 0.113
epoch #18	mean reward = 151.160	epsilon = 0.101
epoch #19	mean reward = 181.230	epsilon = 0.091
epoch #20	mean reward = 175.170	epsilon = 0.082
epoch #21	mea